# Tutorial: APEX for AIME (Math)
In this tutorial, we optimize GPT-4.1 Mini's Chain of Thought (`dspy.ChainOfThought`) for solving math problems (AIME) using the `dspy.APEX` optimizer. APEX performs targeted failure/success analyses, synthesizes hypotheses, and keeps the best prompts observed on the calibration set.

<details>
<summary>Recommended: Set up MLflow Autologging to understand what's happening under the hood.</summary>

### MLflow DSPy Integration

<a href="https://mlflow.org/">MLflow</a> is an LLMOps tool that natively integrates with DSPy and offers explainability and experiment tracking. MLflow's autologging capability automatically tracks progress of APEX optimization, as well as visualizes prompts and module executions as traces to understand DSPy's behavior better. You can set up MLflow easily by following the four steps below.

**Visualize module executions as traces**

![MLflow Trace](./mlflow-tracing-gepa-aime.png)

**Automatically track optimization progress and results**

![MLflow Tracking](./mlflow-tracking-gepa-aime-optimization.png)


**Setup MLflow**

1. Install MLflow

```bash
%pip install mlflow>=3.0.0
```

2. Start MLflow UI in a separate terminal
```bash
mlflow ui --port 5000 --backend-store-uri sqlite:///mlruns.db
```

3. Connect the notebook to MLflow
```python
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("DSPy")
```

4. Enable autologging.

```python
mlflow.dspy.autolog(
    # Log the optimization progress
    log_compiles=True,
    # Log the evaluation results
    log_evals=True,
    # Log traces from module executions
    log_traces=True,
)
```

To learn more about the integration, visit [MLflow DSPy Documentation](https://mlflow.org/docs/latest/llms/dspy/index.html) as well.
</details>

In [2]:
import os
import dspy
from dspy.adapters import JSONAdapter

api_key = 'sk-12345' #input("Enter your OpenAI API key: ")
base_url = "https://nexus-master.lmndstaging.com"
model_prefix = "litellm_proxy"

student_lm = dspy.LM(
    model=f"{model_prefix}/openai/gpt-5-mini",
    api_key=api_key,
    base_url=base_url,
    reasoning_effort="minimal",
    temperature=0.0,
)
analysis_lm = dspy.LM(
    model=f"{model_prefix}/openai/gpt-5",
    api_key=api_key,
    base_url=base_url,
    reasoning_effort="minimal",
    temperature=1.0,
)

# APEX uses JSON adapters by default; exposing them makes customization explicit
analysis_adapter = JSONAdapter()
hypothesis_adapter = JSONAdapter()

n_threads = 50  # notebook thread budget used for evaluation and optimization

dspy.configure(lm=student_lm)
dspy.settings.configure(num_threads=n_threads)

### Loading the AIME dataset

The AIME exam consists of 2 problem sets of size 15 for each year. For this tutorial, we will use AIME problem sets from previous years (2022-2024) for optimization (amounting to total 3 years × 2 sets × 15 problems = 90 problems, split equally between train and validation sets), and test the performance on AIME 2025 (2 sets × 15 problems = 30 problems). Since AIME 2025 is a small set, we repeat it 5 times for statistical stability in evaluation.

In [3]:
from datasets import load_dataset
import random


def init_dataset():
    train_split = load_dataset("AI-MO/aimo-validation-aime")['train']
    train_split = [
        dspy.Example({
            "problem": x['problem'],
            "solution": x['solution'],
            "answer": x['answer'],
        }).with_inputs("problem")
        for x in train_split
    ]
    random.Random(0).shuffle(train_split)
    tot_num = len(train_split)

    test_split = load_dataset("MathArena/aime_2025")['train']
    test_split = [
        dspy.Example({
            "problem": x['problem'],
            "answer": x['answer'],
        }).with_inputs("problem")
        for x in test_split
    ]

    train_set = train_split[: int(0.5 * tot_num)]
    val_set = train_split[int(0.5 * tot_num):]
    test_set = test_split * 5

    return train_set, val_set, test_set

In [4]:
train_set, val_set, test_set = init_dataset()

len(train_set), len(val_set), len(test_set)

(45, 45, 150)

Let's view an example task input

In [5]:
print("Problem:")
print(train_set[0]['problem'])
print("\n\nSolution:")
print(train_set[0]['solution'])
print("\n\nAnswer:")
print(train_set[0]['answer'])

Problem:
In isosceles trapezoid $ABCD$, parallel bases $\overline{AB}$ and $\overline{CD}$ have lengths $500$ and $650$, respectively, and $AD=BC=333$. The angle bisectors of $\angle{A}$ and $\angle{D}$ meet at $P$, and the angle bisectors of $\angle{B}$ and $\angle{C}$ meet at $Q$. Find $PQ$.


Solution:
We have the following diagram:

Let $X$ and $W$ be the points where $AP$ and $BQ$ extend to meet $CD$, and $YZ$ be the height of $\triangle AZB$. As proven in Solution 2, triangles $APD$ and $DPW$ are congruent right triangles. Therefore, $AD = DW = 333$. We can apply this logic to triangles $BCQ$ and $XCQ$ as well, giving us $BC = CX = 333$. Since $CD = 650$, $XW = DW + CX - CD = 16$.
Additionally, we can see that $\triangle XZW$ is similar to $\triangle PQZ$ and $\triangle AZB$. We know that $\frac{XW}{AB} = \frac{16}{500}$. So, we can say that the height of the triangle $AZB$ is $500u$ while the height of the triangle $XZW$ is $16u$. After that, we can figure out the distance from 

### Let's define the program: A simple `dspy.ChainOfThought`

In [6]:
class GenerateResponse(dspy.Signature):
    """Solve the problem and provide the answer in the correct format."""
    problem = dspy.InputField()
    answer = dspy.OutputField()


program = dspy.ChainOfThought(GenerateResponse)

### Defining the evaluation metric
We simply check exact match between the predicted answer and the correct answer.

In [7]:
def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    try:
        llm_answer = int(prediction.answer)
    except ValueError:
        return 0
    return int(correct_answer == llm_answer)

### Evaluating unoptimized Chain Of Thought

We evaluate with the thread budget defined above and tolerate up to `len(test_set)` transient errors so the run completes even on constrained proxies. If your provider enforces stricter limits, lower `n_threads` or tighten `max_errors`.

In [8]:
# Removed max_errors configuration since there should be no errors
eval_kwargs = dict(
    num_threads=n_threads,
    display_progress=True,
    display_table=5,
    provide_traceback=False,
)

evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    **eval_kwargs,
)

baseline_result = evaluate(program)
baseline_result.score

Average Metric: 80.00 / 150 (53.3%): 100%|██████████| 150/150 [00:00<00:00, 637.07it/s]

2025/10/11 23:15:16 INFO dspy.evaluate.evaluate: Average Metric: 80 / 150 (53.3%)


,problem,example_answer,reasoning,pred_answer,metric
0,Find the sum of all integer bases $b>9$ for which $17_b$ is a divi...,70,We interpret 17_b = b+7 and 97_b = 9b+7. We need b+7 to divide 9b+...,70,✔️ [1]
1,"On $\triangle ABC$ points $A, D, E$, and $B$ lie in that order on ...",588,"Set up affine coordinates with A=(0,0), B=(1,0), C=(0,1). Points o...",441,✔️ [0]
2,The 9 members of a baseball team went to an ice-cream parlor after...,16,"We must count assignments of 3 labeled flavors (C, V, S) to 9 dist...",16,✔️ [1]
3,"Find the number of ordered pairs $(x,y)$, where both $x$ and $y$ a...",117,"We need integer solutions (x,y) in [-100,100] satisfying 12x^2 - x...",117,✔️ [1]
4,There are $8!= 40320$ eight-digit positive integers that use each ...,279,Divisibility by 22 means divisible by 2 and 11. Units digit must b...,279,✔️ [1]


53.33

### Augmenting the metric for APEX
APEX benefits from feedback about why predictions fail. We extend the metric to provide textual guidance (and optional worked solutions) that the optimizer can feed into its failure and success analyses.

In [9]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    written_solution = example.get('solution', '')
    try:
        llm_answer = int(prediction.answer)
    except ValueError:
        feedback_text = (
            f"The final answer must be a valid integer and nothing else. You responded with '{prediction.answer}', which couldn't be parsed as an integer."
        )
        feedback_text += f" The correct answer is '{correct_answer}'."
        if written_solution:
            feedback_text += (
                f" Here's the full step-by-step solution:\n{written_solution}\n\n"
                "Reflect on this solution and ensure your final answer is a valid integer when you attempt similar problems."
            )
        return dspy.Prediction(score=0, feedback=feedback_text)

    score = int(correct_answer == llm_answer)
    if score == 1:
        feedback_text = f"Your answer is correct. The correct answer is '{correct_answer}'."
    else:
        feedback_text = f"Your answer is incorrect. The correct answer is '{correct_answer}'."

    if written_solution:
        feedback_text += (
            f" Here's the full step-by-step solution:\n{written_solution}\n\n"
            "Use it to identify the mistakes in your reasoning before trying again."
        )

    return dspy.Prediction(score=score, feedback=feedback_text)

### Optimize the program with `dspy.APEX`

APEX runs targeted analyses over failure and success cases, proposes hypotheses with complete prompt updates, and keeps the best candidate on the calibration set. We limit the budget to a few iterations to keep the tutorial runtime manageable. Use `verbosity` to control logging (`"none"`, `"normal"`, or `"high"`) and `num_threads` to parallelize execution.

In [ ]:
from dspy.teleprompt.apex_optimizer import APEX

# Fixed configuration for parallel execution with enhanced visibility
optimizer = APEX(
    metric=metric_with_feedback,
    analysis_lm=analysis_lm,        # Required parameter
    hypothesis_lm=analysis_lm,       # Optional, defaults to analysis_lm if not provided
    analysis_adapter=analysis_adapter,
    hypothesis_adapter=hypothesis_adapter,
    max_iterations=20,
    num_hypotheses=4,
    num_eval_runs=1,
    train_sample=10,
    success_threshold=1.0,
    convergence_patience=3,
    num_threads=n_threads,           # Using n_threads=50 from configuration
    verbosity="high",                # Enhanced visibility into the optimization process
    seed=42,
)

optimized_program = optimizer.compile(
    student=program,
    trainset=train_set,
    valset=val_set,
)

2025/10/11 23:15:20 INFO dspy.teleprompt.apex_optimizer: APEX: running with num_threads=50
2025/10/11 23:15:20 INFO dspy.teleprompt.apex_optimizer: APEX: Configuration - max_iterations=20, num_hypotheses=1, success_threshold=1.00, convergence_patience=2
2025/10/11 23:15:20 INFO dspy.teleprompt.apex_optimizer: APEX: Using seed=42 for reproducibility
2025/10/11 23:15:20 INFO dspy.teleprompt.apex_optimizer: APEX: Evaluating initial baseline on validation set


Processed 45 / 45 examples: 100%|██████████| 45/45 [00:00<00:00, 598.60it/s]

2025/10/11 23:15:20 INFO dspy.teleprompt.apex_optimizer: APEX: Initial baseline score=0.5111
2025/10/11 23:15:20 INFO dspy.teleprompt.apex_optimizer: APEX: iteration 1 started (train sample=30, val size=45)
2025/10/11 23:15:20 INFO dspy.teleprompt.apex_optimizer: APEX: Sampled 30 training examples from 45 total



Processed 30 / 30 examples: 100%|██████████| 30/30 [00:00<00:00, 432.13it/s]

2025/10/11 23:15:20 INFO dspy.teleprompt.apex_optimizer: APEX: Train evaluation complete - 8 failures, 22 successes


...

2025/10/11 23:25:46 INFO dspy.teleprompt.apex_optimizer: APEX: hypothesis #1 (Add a minimal, high-salience guardrail section to the existing protocol: (a) an explicit 'Do not assume extra structure' checklist, (b) a 'Constraints sanity check' requiring non-degeneracy and verification against all stated conditions, and (c) a 'Case coverage confirmation' line to prevent premature restriction. Keep the rest of the prompt unchanged and require a concise hidden scratch checklist before the final numeric answer.) targeting incorrect_geometric_assumption, missing_constraints, incomplete_reasoning [impact=0.72, generalizability=0.78]
2025/10/11 23:25:46 INFO dspy.teleprompt.apex_optimizer:   → predict: You are solving contest-style math problems. Follow this brief, structured protocol and then give only the final numeric answer in the required format.

Protocol:
1) Plan (1-2 sentences): Identify the...
2025/10/11 23:25:46 INFO dspy.teleprompt.apex_optimizer:      Rationale: A short pre-answer 

Processed 45 / 45 examples: 100%|██████████| 45/45 [00:00<00:00, 1135.40it/s]

2025/10/11 23:25:46 INFO dspy.teleprompt.apex_optimizer: APEX: iteration 3 baseline score=0.5111



Processed 45 / 45 examples: 100%|██████████| 45/45 [02:00<00:00,  2.68s/it]

2025/10/11 23:27:46 INFO dspy.teleprompt.apex_optimizer: APEX: iteration 3 hypothesis score=0.3556
2025/10/11 23:27:46 INFO dspy.teleprompt.apex_optimizer: APEX: hypothesis details → {'observation': 'Frequent errors stem from premature assumptions and missing/implicit constraints: the model imposes unjustified symmetry/affinity in geometry, over-restricts cases in counting, and accepts degenerate boundary cases (zero/empty) that violate intended constraints. These lead to incomplete reasoning and wrong final numbers despite otherwise competent algebra on successful tasks.', 'fixable_root_causes': ['incorrect_geometric_assumption', 'missing_constraints', 'incomplete_reasoning'], 'non_fixable_root_causes': [], 'impact_score': 0.72, 'generalizability_score': 0.78, 'strategy': "Add a minimal, high-salience guardrail section to the existing protocol: (a) an explicit 'Do not assume extra structure' checklist, (b) a 'Constraints sanity check' requiring non-degeneracy and verification against 

### Inspect the APEX-optimized prompt

In [11]:
print(optimized_program.predict.signature.instructions)

You are solving contest-style math problems. Follow this brief, structured protocol and then give only the final numeric answer in the required format.

Protocol:
1) Plan (1-2 sentences): Identify the key approach and main constraints.
2) Work (scratch, concise): Do necessary derivations. Keep it short.
3) Check (1-3 bullets, must do):
   - Constraints: Verify all given constraints are satisfied; exclude degenerate/empty or zero cases unless explicitly allowed.
   - Assumptions: Do NOT assume symmetry, parallelism, collinearity, regularity, affinity, or integrality unless stated or proved.
   - Coverage: If cases are involved, confirm that considered cases exhaust all possibilities required by the problem.
4) Final: Output only the required final numeric answer.

Important:
- No extra commentary in the Final; print only the number or exact expression as required.
- If multiple candidates arise, select the one that satisfies all constraints and maximal/minimal conditions as asked.


### Evaluating the Chain Of Thought optimized with APEX

In [12]:
evaluate(optimized_program)

Average Metric: 60.00 / 133 (45.1%):  88%|████████▊ | 132/150 [00:45<00:18,  1.01s/it]

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='[[ ## co...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


Average Metric: 64.00 / 150 (42.7%): 100%|██████████| 150/150 [02:09<00:00,  1.16it/s]

2025/10/11 23:29:56 INFO dspy.evaluate.evaluate: Average Metric: 64 / 150 (42.7%)


,problem,example_answer,reasoning,pred_answer,metric
0,Find the sum of all integer bases $b>9$ for which $17_b$ is a divi...,70,Plan: Interpret 17_b = 1*b +7 = b+7 and 97_b = 9*b +7. Need intege...,70,✔️ [1]
1,"On $\triangle ABC$ points $A, D, E$, and $B$ lie in that order on ...",588,Plan: Use coordinates. Place A at origin along AB horizontally? Be...,588,✔️ [1]
2,The 9 members of a baseball team went to an ice-cream parlor after...,16,Plan: Count number of surjections from 9 labeled players to three ...,292,✔️ [0]
3,"Find the number of ordered pairs $(x,y)$, where both $x$ and $y$ a...",117,Plan: Solve quadratic in x/y factorization. Count integer pairs (x...,117,✔️ [1]
4,There are $8!= 40320$ eight-digit positive integers that use each ...,279,Plan: Count 8-digit permutations of digits 1–8 divisible by 22. Di...,279,✔️ [1]


EvaluationResult(score=42.67, results=<list of 150 results>)

APEX typically improves the GPT-4.1 Mini's performance on AIME 2025 by leveraging targeted analyses while keeping the overall evaluation flow identical to the GEPA tutorial.